In [1]:

from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

/Users/ayushghimire/Documents/GitHub/jupyter-lab-models/arima/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [2]:


import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import mlflow
from statsmodels.tsa.stattools import pacf
from statsmodels.graphics.tsaplots import plot_pacf
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
import  pickle




df = pd.read_csv("./ElectronicsProductsPricingData.csv", encoding='latin1')



df["prices.dateSeen"] = pd.to_datetime(df["prices.dateSeen"], errors='coerce')


# compute a percent‑discount column from the price fields
# discount_percent = ((regular_price - sale_price) / regular_price) * 100
# assume amountMax is the non‑sale price and amountMin the current price;
# guard against division by zero.

df['discount_percent'] = np.where(
    df['prices.amountMax'] > 0,
    100 * (df['prices.amountMax'] - df['prices.amountMin']) / df['prices.amountMax'],
    0
)



/var/folders/st/r78_rxtx0hd_yr_r8qtl08_c0000gn/T/ipykernel_29421/2846710072.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["prices.dateSeen"] = pd.to_datetime(df["prices.dateSeen"], errors='coerce')


In [4]:
df['day_of_week'] = df['prices.dateSeen'].dt.dayofweek
df['month'] = df['prices.dateSeen'].dt.month
df['week_of_year'] = df['prices.dateSeen'].dt.isocalendar().week.astype(int)




# Binary flag for Thursday updates
df['is_thursday'] = (df['day_of_week'] == 3).astype(int)


# Preview the engineered features
df[['prices.dateSeen', 'day_of_week', 'week_of_year', 'is_thursday']].head()


ValueError: cannot convert NA to integer

In [ ]:
daily = df.groupby('prices.dateSeen').agg(y = ("discount_percent", "mean")).reset_index()

## step 2 : - train/test split

train = df[df["prices.dateSeen"].dt.year == 2017]
test = df[df["prices.dateSeen"].dt.year == 2018]


## step 3 : - rename the dateseen column to ds and the discount_percent column to y
train = train.rename(columns={"prices.dateSeen": "ds", "discount_percent": "y"})

## 